# 🛣️ Tutorial 1: Computer Vision Lane Detection Pipeline
**Author:** Nivesh Kumar Meena  
**Project:** AI-Powered Autonomous Self-Driving Car

This notebook demonstrates the end-to-end computer vision pipeline used to extract road lanes, compute vehicle lateral displacement (Cross-Track Error), and estimate curvature radius.

## 1. Overview of the Pipeline
1. **Camera Calibration & Distortion Correction**
2. **Color Filtering & Edge Extraction (HSV Space)**
3. **Perspective Transform (Bird's-Eye View / IPM)**
4. **Sliding Window Histogram Peak Search**
5. **2nd-Order Polynomial Fitting ($x = ay^2 + by + c$)**
6. **Cross-Track Error (CTE) & Road Curvature Calculation**

In [ ]:
import math
import sys
sys.path.append('..')

from src.perception.lane_detector import LaneDetector, LaneDetectionResult
from src.config import CameraConfig
from src.utils.math_utils import fit_quadratic, calculate_curvature

print("Lane detection dependencies imported successfully!")

## 2. Polynomial Fitting on Detected Lane Pixels
In top-down bird's-eye coordinates, a road curve is modeled as a parabola:
$$x(y) = a \cdot y^2 + b \cdot y + c$$
Where:
- $a$ determines lane curvature
- $b$ determines the tangent heading angle
- $c$ is the lateral position offset at $y = 0$

In [ ]:
# Synthetic lane coordinates (approaching a right turn)
y_points = [100.0, 200.0, 300.0, 400.0, 480.0]
x_points = [320.0, 330.0, 350.0, 380.0, 410.0]

a, b, c = fit_quadratic(x_points, y_points)
print(f"Fitted Polynomial: x(y) = {a:.6f}*y^2 + {b:.4f}*y + {c:.2f}")

radius_m = calculate_curvature(a, b, y_eval=480.0)
print(f"Calculated Road Curvature Radius: {radius_m:.2f} meters")

## 3. Lateral Error (CTE) Calculation
$$\text{Error} = \text{Car Center} - \text{Lane Center}$$
This error is directly passed to the PID lateral steering controller.

In [ ]:
lane_detector = LaneDetector()
sample_synthetic_frame = {"cross_track_error": 0.12, "heading_error": 0.05, "curvature": 4.5}
res = lane_detector.process_frame(sample_synthetic_frame)
print(f"Lane Detected:      {res.detected}")
print(f"Cross-Track Error:  {res.cross_track_error_m:.3f} m")
print(f"Heading Error:      {math.degrees(res.heading_error_rad):.2f} deg")
print(f"Confidence:         {res.confidence * 100:.1f}%")